# Cài các thư viện cần thiết

In [ ]:
!pip install fastapi uvicorn nest-asyncio python-multipart transformers torch pillow


Cấu hình 

In [1]:
yaml_config =  """
               model_path: "Salesforce/blip-vqa-base"
               """
with open("config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_config)

build model

In [ ]:
from transformers import BlipProcessor, BlipForQuestionAnswering
from omegaconf import OmegaConf
import torch
from PIL import Image

class VisualQuestionAnswering:
    def __init__(self, config_path):
        self.config = OmegaConf.load(config_path)

        self.processor = BlipProcessor.from_pretrained(self.config.model_path)
        self.model = BlipForQuestionAnswering.from_pretrained(self.config.model_path)

    def __call__(self, image: Image.Image, query: str):
        inputs = self.processor(image, query, return_tensors="pt")

        with torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=50)

        return self.processor.decode(out[0], skip_special_tokens=True)

init model

In [ ]:
vqa_model = VisualQuestionAnswering("config.yaml")

init api

In [ ]:
import torch
import nest_asyncio
import uvicorn
import threading
import io
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware

from PIL import Image

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

@app.get("/")
async def root():
    return {"message": "API có khả năng đọc và trả lời các câu hỏi liên quan đến ảnh đầu vào"}



@app.get("/health")
async def health_check():
    return {"status": "Hệ thống hoạt động bình thường"}

@app.post('/predict')
async def predict(file: UploadFile = File(...), query: str = Form(default="")):
    try:
        contents = await file.read()
        image = Image.open(io.BytesIO(contents)).convert("RGB")

        result = vqa_model(image, query=query)

        return {
            "answer": result
        }
    except Exception as e:
        return {"error": str(e)}




run server

In [ ]:
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

nest_asyncio.apply()
thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print("Server đang chạy tải tại http://0.0.0.0:8000 (Local)")

Chạy lệnh bên dưới để tạo public api 

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

!nohup cloudflared tunnel --url http://127.0.0.1:8000 > cloudflared.log 2>&1 &
!sleep 5
!grep -o 'https://.*\.trycloudflare.com' cloudflared.log

Hoặc chạy lệnh sau ở terminal, không được nhấn Ctr+C vì sẽ tắt server

In [ ]:
#!ssh -p 443 -R0:localhost:8000 qr@a.pinggy.io
